In [34]:
from mne.decoding import CSP
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras.losses import CategoricalCrossentropy
from network import NetCNN1D, NetCNN2D_CSP
from sklearn import metrics

import numpy as np
import sys, os

import dataset_NewEEG
from config_NewEEG import Config

import matplotlib.pyplot as plt


import pandas as pd



config = Config()



# Choose from: 'CLeft', 'CRight', 'CUp' and 'CDown'
config.used_classes = [['CLeft'], ['CRight']]
config.session_type = 'S2'


config.t_start = 0 
config.t_end = 2.5

config.window_size = 2.0
config.stride = 0.2



config.n_csp_components = 3

# All available channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']
config.used_channels = ['FC3', 'FCz', 'FC4', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CPz', 'CP4']

In [35]:
config.used_classes = [['CLeft'], ['CRight']]#, ['CUp'], ['CDown']]
config.session_type = 'S2'


X, y = dataset_NewEEG.session_dataset(config ,'I09')

(25, 13, 625)
(25,)


In [37]:

crossValidation_KF = StratifiedKFold(n_splits=8, shuffle=True, random_state=0)

shuffle_indices = np.random.permutation(len(y))

    
X = X[shuffle_indices]
y = y[shuffle_indices]


acc = []
Y_preds = []

for train_index, valid_index in crossValidation_KF.split(X,y):
    X_train = X[train_index]
    Y_train = y[train_index]
    X_valid = X[valid_index]
    Y_valid = y[valid_index]

    X_train, Y_train = dataset_NewEEG.slice_EEG_epoch(config, X_train, Y_train)
    X_valid, Y_valid = dataset_NewEEG.slice_EEG_epoch(config, X_valid, Y_valid)


    m = X_train.mean()
    sig = X_train.std()

    X_train = (X_train - m)/sig
    X_valid = (X_valid - m )/sig
    

    lda = LDA()
    csp = CSP(n_components=config.n_csp_components , reg=None, log=True, norm_trace=False, transform_into='average_power')

    X_train = csp.fit_transform(X_train, Y_train)
    X_valid = csp.transform(X_valid)

    

    lda.fit(X_train, Y_train)

    acc.append(lda.score(X_valid, Y_valid))
    confusion_matrix = metrics.confusion_matrix(Y_valid, lda.predict(X_valid))




print("\n\n ================================== \n")
print(acc)
print(f"{100*np.mean(acc):.2f}% +- {100*np.std(acc)}")

Computing rank from data with rank=None
    Using tolerance 1.3 (2.2e-16 eps * 13 dim * 4.6e+14  max singular value)
    Estimated rank (data): 13
    data: rank 13 computed from 13 data channels with 0 projectors
Reducing data rank from 13 -> 13
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.4 (2.2e-16 eps * 13 dim * 4.8e+14  max singular value)
    Estimated rank (data): 13
    data: rank 13 computed from 13 data channels with 0 projectors
Reducing data rank from 13 -> 13
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.4 (2.2e-16 eps * 13 dim * 4.7e+14  max singular value)
    Estimated rank (data): 13
    data: rank 13 computed from 13 data channels with 0 projectors
Reducing data rank from 13 -> 13
Estimating class=0 covariance using EMPIRICAL
Done.